# Tutorial on Jupyter, Colab, NNs and INRs.

Welcome! This notebook walks you through three topics:

1. **Setup** — Connect GitHub to Colab, run your first cells, and turn on the GPU.
2. **Neural Networks** — What is a neuron? What is training? How does an MLP learn a function?
3. **Implicit Neural Representations** — Use a coordinate network to fit a real image, discover the spectral bias problem, and learn the techniques the research community developed to overcome it: positional encoding and SIREN.

By the end you will have trained three different networks on the same image and understand *why* each one performs differently.

---
# Part 1: General Setup & First Steps.

## Task 1.1: Connect GitHub to Google Colab

Google Colab is a free, cloud-based Python environment that gives you access to hardware — including GPUs — at no cost [[1](https://colab.research.google.com/)][[2](https://research.google.com/colaboratory/faq.html)].

To keep your work safe and shareable, we link it to GitHub:

1. Create a new repository on your GitHub account.
2. Add a new file to it named `tutorial.ipynb` (the `.ipynb` extension marks it as a Jupyter Notebook).
3. Go to [Google Colab](https://colab.research.google.com/).
4. On the welcome screen, click the **GitHub** tab.
5. Search for your GitHub username and select your repository.
6. Click on `tutorial.ipynb` to open it.

**Tip**: Save your work back to GitHub anytime via **File → Save a copy in GitHub**.

In [ ]:
%ls
%pwd

## Task 1.2: Running Jupyter Cells

A Jupyter Notebook is made of **cells**. There are two kinds:

- **Markdown cells** — formatted text (like this one). Double-click to edit.
- **Code cells** — Python code. Run them to execute.

### How to run a cell

| What you want to do | Keyboard Shortcut |
| :--- | :--- |
| Run the cell and jump down to the next one | `Shift + Enter` |
| Run the cell but stay right where you are | `Ctrl + Enter` |

### Magic commands

Inside code cells, special commands starting with `%` are called **magic commands**. They talk to the IPython interpreter, not to Python itself.

| Command | What it does |
|---------|-------------|
| `%matplotlib inline` | Show plots directly below the cell |
| `%%time` | Measure how long the whole cell takes |
| `%who` | List all variables currently defined |
| `%pip install <pkg>` | Install a package inside the active environment |

**Question:** What is the difference between `%pip install` and `!pip install`?

- **`%pip install`** is a magic command. It installs the package into the exact Python environment that the current Jupyter kernel is using. This is the recommended approach in Colab.
- **`!pip install`** runs a shell command. It *usually* works, but can silently install into a different environment and cause version conflicts.

**Rule of thumb:** Always use `%pip install` in Colab/Jupyter.

In [ ]:
%who

## Task 1.3: Install Required Libraries

Run the cell below. It installs everything this notebook needs.

- **torch** — PyTorch, the deep learning framework we will use throughout.
- **matplotlib** — For plotting graphs and displaying images.
- **numpy** — Numerical arrays (PyTorch tensors are similar, but numpy is handy for quick math).
- **pillow** — Loading and saving images.
- **requests** — Downloading files from URLs.

Press `Shift + Enter` to run the cell and wait until you see `Successfully installed...` (or `already satisfied`).

In [ ]:
%pip install torch matplotlib numpy pillow requests

## Task 1.4: Activate the GPU

To enable the GPU in Colab:
1. Click **Runtime** in the top menu.
2. Select **Change runtime type**.
3. Under **Hardware accelerator**, choose **GPU** (T4 is fine).
4. Click **Save**.

Then run the cell below to confirm it is working.

In [ ]:
import torch

# This tells PyTorch to use the GPU if it's available, otherwise fall back to CPU
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

print(f"You are currently using: {device.upper()}")

## Task 1.5: Connect to Google Drive

Colab sessions are **temporary** — when the runtime disconnects, any files you saved locally are lost. Connecting Google Drive gives you persistent storage that survives session restarts.

This is useful for:
- Saving trained model weights so you can reload them later.
- Loading large datasets without re-downloading them every session.
- Keeping intermediate results and plots.

1. Run the cell below.
2. A pop-up will ask you to choose a Google account and grant access.
3. Click **Connect to Google Drive** and follow the prompts.

Once mounted, your Drive is accessible at `/content/drive/MyDrive/`.

In [ ]:
from google.colab import drive

# Mount Google Drive at /content/drive
drive.mount('/content/drive')

---
# Part 2: Neural Networks

This part is aimed at building your intuition for how neural networks work. What a neuron is, how training happens, and how an MLP can learn a smooth function from data.

## Neural Networks

### Artificial Neuron

The fundamental unit of the network is the neuron. It processes information, taking multiple numerical inputs, scaling each by a learned weight, summing them together with a constant bias, and passing the final sum through an activation function.

```
output = activation(w₁·x₁ + w₂·x₂ + ... + wₙ·xₙ + b)
```

The activation function introduces non-linearity. Without it, the entire network—regardless of its size—would mathematically collapse into a single, rigid linear equation, completely stripping away its ability to model complex patterns.

### Stacking neurons into layers
Individual neurons are organized into sequential layers, connected by pathways that each carry a unique, adjustable weight. In deep neural nets such layers are stacked according to the complexity and dimensionality of the problem. According to [the Universal Approximation Theorem](https://www.deep-mind.org/2023/03/26/the-universal-approximation-theorem/), a network equipped with at least one hidden layer—and a sufficient number of neurons—possesses the mathematical flexibility to approximate any continuous function.

### Common activation functions

**Question:** What does an activation function do, and why is ReLU so popular?

<details>
<summary><b>Click to see the answer</b></summary>
<br>

An activation function breaks linearity. Without it, `W₂(W₁x + b₁) + b₂` simplifies to a single matrix multiply — just a linear transform no matter how many layers you stack.

**ReLU** (Rectified Linear Unit) simply outputs `max(0, x)`. It is popular because:
- It is very fast to compute.
- Its gradient is either 0 or 1, which avoids the *vanishing gradient* problem that plagued earlier activations like sigmoid.
- Even though a single ReLU is piecewise-linear, a network combining thousands of them can flexibly mold to fit highly complex, non-linear data boundaries.

</details>

### How does a network learn?
0. **Computational Graph** — construct a computational graph to calculate all partial derivatives using automatic differentiation (autodiff).
1. **Forward pass** — feed input through the network to get a prediction.
2. **Compute loss** — measure how wrong the prediction is (e.g. Mean Squared Error).
3. **Backward pass** — use the backpropagation algorithm to compute how each weight contributed to the error.
4. **Update weights** — nudge each weight in the direction that reduces the loss (gradient descent).

Repeat thousands of times until the loss is small.

Here is a [tutorial on backpropagation and autodiff](https://www.youtube.com/watch?v=SmZmBKc7Lrs)

For more intuition on neural networks feel free to read [the XOR problem story](https://towardsdatascience.com/how-neural-networks-solve-the-xor-problem-59763136bdd7/)

## Task 2.1: Train Your First MLP

Now, you'll train a small **MLP** (Multi-Layer Perceptron) to learn the function `y = x²`.

The network takes a single number `x` as input and should output `x²`. It has no idea what the function is — it will figure it out purely from examples.

**Your job:** Fill in the three `### STUDENT CODE HERE ###` sections.

1. **Define the model** — a `nn.Sequential` with two hidden layers of size 64 and ReLU activations, ending in a single output neuron.
2. **Compute the loss** — MSE between the model's prediction and the true value.
3. **Update the weights** — call the optimizer to take a gradient step.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)

# 1. Create training data: 1000 points of y = x².
x_train = torch.linspace(-1, 1, 1000).unsqueeze(1).to(device)
y_train = x_train ** 2

# 2. Define the model: nn.Sequential, 2 hidden layers of size 64, ReLU activations, 1 output.
### STUDENT CODE HERE ###
model = ...
model = model.to(device)

# 3. Set up the optimizer.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []

for step in range(2000):
    pred = model(x_train)

    # Compute MSE loss between pred and y_train.
    ### STUDENT CODE HERE ###
    loss = ...

    optimizer.zero_grad()
    loss.backward()
    # Take an optimizer step.
    ### STUDENT CODE HERE ###
    ...

    losses.append(loss.item())

# 4. Plot the training curve and the learned function.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(losses)
ax1.set_xlabel("Training step")
ax1.set_ylabel("MSE Loss")
ax1.set_title("Training loss")
ax1.set_yscale("log")

with torch.no_grad():
    x_test = torch.linspace(-1, 1, 200).unsqueeze(1).to(device)
    y_pred = model(x_test).cpu()

ax2.plot(x_test.cpu(), x_test.cpu() ** 2, label="True: y = x²", linewidth=2)
ax2.plot(x_test.cpu(), y_pred, "--", label="MLP prediction", linewidth=2)
ax2.legend()
ax2.set_title("Learned function vs ground truth")
plt.tight_layout()
plt.show()

Your MLP approximates `y = x²` well because it is a smooth, low-frequency function. In Part 3 you will discover what happens when the signal contains fine detail — and learn the techniques the research community developed to fix it.

---
# Part 3: Implicit Neural Representations

An **Implicit Neural Representation (INR)** is a neural network used as a continuous signal function — it maps coordinates to signal values instead of storing them in a grid. In this part you will fit a real image with three progressively more capable INRs, discover the core obstacles from the literature, and measure improvement with PSNR.

*We treat is as continuous function but there are methods to learn discontinuous functions by treating them as piece-wise continuous ones*

## What is an Implicit Neural Representation (INR)?

### The idea

Normally, a digital image is stored as a **grid of pixels** — a 2D array of (R, G, B) triples. This is **explicit**, the signal values are stored directly per grid cell.

An **Implicit Neural Representation** takes a completely different approach. Instead of storing pixels, we train a neural network `f` such that:
```
f(x, y)  →  (R, G, B)
```
Given any continuous coordinate `(x, y)`, the network outputs the colour at that location. **The network weights are the image** — there is no pixel grid at all.

### Why are they useful?
- **Infinite Resolution** - Standard images pixelate when zoomed, but INRs use continuous mathematical functions to generate smooth, crisp details at any scale.

- **Fixed Storage Size** - Traditional files grow with pixel count. INR storage is tied only to the neural network's size, so rendering at higher resolutions doesn't increase the file size on your drive.

- **Fully Differentiable** - Unlike discrete pixel grids, INRs allow for continuous gradient calculations. This is crucial for running physics simulations, optimizing models, and generating 3D scenes.

- **Inherent Compression** - Instead of relying on lossy algorithms like JPEG, INRs naturally compress data by learning and storing the underlying mathematical patterns rather than every individual pixel.


### Applications
INRs have become a foundational tool in modern 3D vision and physics:

- **3D shape representation** — store a 3D object as a signed distance function learned by a network: [[DeepSDF (Park et al. 2019)](https://arxiv.org/abs/1901.05103)], [[Occupancy Networks (Mescheder et al. 2019)](https://arxiv.org/abs/1812.03828)]
- **Novel view synthesis** — reconstruct a 3D scene from 2D photos and render it from any angle: [[NeRF (Mildenhall et al. 2020)](https://arxiv.org/abs/2003.08934)]
- **Physics simulation** — represent continuous fields (temperature, velocity) [[PEINR (Shen et al. 2025)](https://openreview.net/pdf?id=1QZMWVrgsU)]

**Question:** In the image INR setting, what are the inputs and outputs of the network?
<details>
<summary><b>Click to see the answer</b></summary>
<br>

- **Inputs:** normalised pixel coordinates `(x, y) ∈ [-1, 1]²`
- **Outputs:** colour values `(R, G, B) ∈ [0, 1]³`
Every pixel in the image becomes one training example. We show the network every coordinate, compare its colour prediction to the true pixel colour, and train with MSE loss.
The standarization to `[-1,1]` and `[0, 1]` has become a standard practice, and a result of experiments on the topic [[Efficient BackProp (LeCun et al. 1998)](https://cseweb.ucsd.edu/classes/wi08/cse253/Handouts/lecun-98b.pdf)]
</details>

## The Spectral Bias Problem

INRs must represent fine detail — sharp edges, textures, high-frequency patterns. But plain ReLU networks have a fundamental obstacle.

Let's try to fit `y = sin(20πx)` using the same architecture from Task 2.1.

Run the cell below *without changes* and observe the result.

**Question:** Why does the ReLU network fail to learn this high-frequency signal?

<details>
<summary><b>Click to see the answer</b></summary>
<br>

Neural networks with standard activations (ReLU, tanh, sigmoid) have a built-in bias toward learning **low-frequency components first**. This is called **spectral bias** or the *frequency principle*, you can read about the problem in this publication:

> Rahaman, N. et al. (2019). *On the Spectral Bias of Neural Networks*. [[arxiv](https://arxiv.org/abs/1806.08734)]

Intuitively: a small nudge to any weight produces a smooth, low-frequency change in the network's output. To represent a rapidly oscillating function, many weights need to combine in a very precise way — something that gradient descent struggles to coordinate.

The result is a network that converges to a blurry, low-frequency approximation and stops improving. This is the core problem that Tasks 3.4 and 3.5 solve.
</details>

In [ ]:
import math

torch.manual_seed(42)

# 1. Create the high-frequency target: sin(20πx).
x_train_hf = torch.linspace(-1, 1, 1000).unsqueeze(1).to(device)
y_train_hf = torch.sin(20 * math.pi * x_train_hf)

# 2. Use the same architecture as Task 2.1.
model_hf = nn.Sequential(
    nn.Linear(1, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 1)
).to(device)

# 3. Train.
optimizer_hf = torch.optim.Adam(model_hf.parameters(), lr=1e-3)
losses_hf = []

for step in range(2000):
    pred = model_hf(x_train_hf)
    loss = nn.functional.mse_loss(pred, y_train_hf)
    optimizer_hf.zero_grad()
    loss.backward()
    optimizer_hf.step()
    losses_hf.append(loss.item())

# 4. Plot and compare the result to the true signal.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(losses_hf)
ax1.set_xlabel("Training step")
ax1.set_ylabel("MSE Loss")
ax1.set_title("Training loss (high-frequency target)")
ax1.set_yscale("log")

with torch.no_grad():
    x_test = torch.linspace(-1, 1, 500).unsqueeze(1).to(device)
    y_pred_hf = model_hf(x_test).cpu()
    y_true_hf = torch.sin(20 * math.pi * x_test).cpu()

ax2.plot(x_test.cpu(), y_true_hf, label="True: sin(20πx)", alpha=0.7)
ax2.plot(x_test.cpu(), y_pred_hf, "--", label="MLP prediction", alpha=0.7)
ax2.legend()
ax2.set_title("MLP vs high-frequency signal")
plt.tight_layout()
plt.show()

print(f"Final loss: {losses_hf[-1]:.6f}  (compare to Task 2.1's loss!)")

## Task 3.1: Fit an Image with a Naive MLP

You saw spectral bias in 1D in the previous cell's output. Now let's see it in 2D — on a real image.

We will:
1. Load a small reference image from the internet.
2. Turn every pixel coordinate into a training example.
3. Train a plain ReLU MLP to map coordinates → colours.
4. Reconstruct the image from the network and compare.

**Your job:** Fill in the three `### STUDENT CODE HERE ###` sections.

1. **Normalise coordinates** — map pixel indices `(row, col)` to the range `[-1, 1]`.
2. **Define the MLP** — 2 inputs (x, y), 3 outputs (R, G, B), at least two hidden layers of width 256.
3. **Reconstruct the image** — query the trained network at all coordinates and reshape to `(H, W, 3)`.

In [ ]:
import requests
import numpy as np
from PIL import Image
from io import BytesIO

torch.manual_seed(42)

# 1. Download and resize the image to 128×128.
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
response = requests.get(url)
img_pil = Image.open(BytesIO(response.content)).convert("RGB").resize((128, 128))
img_np = np.array(img_pil, dtype=np.float32) / 255.0  # values in [0, 1]

H, W, C = img_np.shape
print(f"Image shape: {H}×{W}×{C}")

# 2. Build a pixel coordinate grid and normalise to [-1, 1].
rows = torch.arange(H, dtype=torch.float32)
cols = torch.arange(W, dtype=torch.float32)
grid_y, grid_x = torch.meshgrid(rows, cols, indexing="ij")

# Divide by (H-1) or (W-1) to reach [0, 1], then scale to [-1, 1].
### STUDENT CODE HERE ###
coords = torch.stack([..., ...], dim=-1).reshape(-1, 2).to(device)  # (H*W, 2)

colors = torch.tensor(img_np, dtype=torch.float32).reshape(-1, 3).to(device)  # (H*W, 3)

# 3. Define a ReLU MLP: 2 inputs, two hidden layers of size 256, 3 outputs.
### STUDENT CODE HERE ###
naive_mlp = ...
naive_mlp = naive_mlp.to(device)

# 4. Train the network.
optimizer = torch.optim.Adam(naive_mlp.parameters(), lr=1e-3)
naive_losses = []

for step in range(2000):
    pred = naive_mlp(coords)
    loss = nn.functional.mse_loss(pred, colors)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    naive_losses.append(loss.item())
    if step % 500 == 0:
        print(f"Step {step:4d} | Loss: {loss.item():.6f}")

# 5. Reconstruct the image by querying the network at every coordinate.
naive_mlp.eval()
with torch.no_grad():
    ### STUDENT CODE HERE ###
    recon_naive = ...  # numpy array, shape (H, W, 3), clipped to [0, 1]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(img_np)
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(recon_naive)
axes[1].set_title("Naive MLP")
axes[1].axis("off")

plt.tight_layout()
plt.show()

naive_recon_saved = recon_naive.copy()  # kept for comparison in next cells

## Task 3.2: Positional Encoding (Fourier Features)

The blurry result from Task 3.1 is spectral bias in action on 2D data. The network's raw inputs are just two numbers `(x, y)` — small, slowly-varying scalars. The network has no efficient way to construct high-frequency outputs from them.

### The fix: lift coordinates into a high-dimensional space

**Positional encoding** maps each coordinate through a bank of sinusoids at increasing frequencies:

```
γ(x) = [sin(2⁰πx), cos(2⁰πx), sin(2¹πx), cos(2¹πx), ..., sin(2^(L-1)πx), cos(2^(L-1)πx)]
```

Applied to both `x` and `y`, a coordinate pair `(x, y)` becomes a vector of length `4L`. The MLP now sees a larger space that can map to many different frequencies and can combine them to represent fine detail.

This technique was introduced in the context of Neural Radiance Fields [[NeRF (Mildenhall et al. 2020)](https://arxiv.org/abs/2003.08934)] and studied in depth as *Fourier feature networks* [[Tancik et al. 2020](https://arxiv.org/abs/2006.10739)].

**Question:** Why does adding sinusoids at many frequencies help overcome spectral bias?

<details>
<summary><b>Click to see the answer</b></summary>
<br>

Without encoding, the network receives smooth, low-variation inputs. Even if it wanted to produce a high-frequency output, the gradient signal would be very weak — the loss landscape is almost flat with respect to the weights that would create high-frequency variation.

After encoding, the input already contains components at many frequencies. The MLP just needs to learn how to *mix* them — a much easier optimisation problem. Essentially, we have transferred the hard task of creating frequencies from the MLP to a fixed, known transformation.

The key insight from Tancik et al. is that standard MLPs are **low-pass filters** in input space, and encoding overcomes this by providing the network with high-frequency basis functions as ready-made inputs.

</details>

**Your job:** Implement the `PositionalEncoding` class below, then train a new MLP on top of it and compare the reconstruction to Task 3.1.

In [ ]:
torch.manual_seed(42)

L = 10  # number of frequency bands


class PositionalEncoding(nn.Module):
    """Maps (x, y) ∈ [-1,1]² to a 4L-dimensional Fourier feature vector."""

    def __init__(self, num_frequencies: int):
        super().__init__()
        self.num_frequencies = num_frequencies

    def forward(self, coords: torch.Tensor) -> torch.Tensor:
        # For each k in [0, ..., L-1]: append sin(2^k·π·coords) and cos(2^k·π·coords).
        ### STUDENT CODE HERE ###
        encoded = []
        for k in range(self.num_frequencies):
            ...
        return torch.cat(encoded, dim=-1)


# 1. Build the encoding layer and the MLP on top of it.
pe = PositionalEncoding(L).to(device)
input_dim = 4 * L  # two coordinates × two sinusoids × L frequency bands

pe_mlp = nn.Sequential(
    nn.Linear(input_dim, 256), nn.ReLU(),
    nn.Linear(256, 256), nn.ReLU(),
    nn.Linear(256, 3)
).to(device)

# 2. Train.
optimizer_pe = torch.optim.Adam(pe_mlp.parameters(), lr=1e-3)
pe_losses = []

for step in range(2000):
    encoded_coords = pe(coords)
    pred = pe_mlp(encoded_coords)
    loss = nn.functional.mse_loss(pred, colors)
    optimizer_pe.zero_grad()
    loss.backward()
    optimizer_pe.step()
    pe_losses.append(loss.item())
    if step % 500 == 0:
        print(f"Step {step:4d} | Loss: {loss.item():.6f}")

# 3. Reconstruct and compare to the naive MLP.
pe_mlp.eval()
with torch.no_grad():
    recon_pe = pe_mlp(pe(coords)).clamp(0, 1).cpu().numpy().reshape(H, W, 3)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(img_np);           axes[0].set_title("Original");  axes[0].axis("off")
axes[1].imshow(naive_recon_saved); axes[1].set_title("Naive MLP"); axes[1].axis("off")
axes[2].imshow(recon_pe);          axes[2].set_title("PE-MLP");    axes[2].axis("off")
plt.tight_layout()
plt.show()

pe_recon_saved = recon_pe.copy()  # kept for comparison in next cells

## Task 3.3: SIREN — Sinusoidal Representation Networks

Positional encoding improves quality by pre-processing the inputs. **SIREN** takes a fundamentally different approach, it changes the *activation function* inside the network itself [[Sitzmann et al. 2020](https://arxiv.org/abs/2006.09661)].

Instead of ReLU, every neuron uses sine - `sin` in torch.

```
output = sin(W·x + b)
```

### Why sinusoidal activations?

1. **Derivatives are also sinusoidal** — a SIREN's gradient with respect to its inputs is itself a SIREN. This makes SIRENs ideal for tasks involving derivatives (e.g. solving PDEs, computing surface normals).
2. **Natural affinity for oscillatory signals** — sine activated neurons can represent high-frequency content directly, without needing an explicit encoding step.
3. **Compositionality** — `sin(sin(x))` still has rich spectral content, unlike `relu(relu(x))` which just clips.

### The critical initialisation scheme

SIREN needs a careful weight initialisation to work. If weights are too large, early layers saturate; if too small, gradients vanish.

Sitzmann et al. derived that:
- **First layer:** weights drawn uniformly from `[-1/n_in, 1/n_in]`
- **Hidden layers:** weights drawn uniformly from `[-√(6/n_in), √(6/n_in)]`

where `n_in` is the number of inputs to that layer. An additional frequency hyperparameter **ω₀** (typically 30) scales the first layer's weights to set the "base frequency".

**Your job:** Implement `SIRENLayer` (a linear layer followed by `sin`) with the correct initialisation.

> Sitzmann, V. et al. (2020). *Implicit Neural Representations with Periodic Activation Functions*. NeurIPS. [[arxiv](https://arxiv.org/abs/2006.09661)]

In [ ]:
torch.manual_seed(42)

OMEGA_0 = 30.0  # base frequency for the first layer


class SIRENLayer(nn.Module):
    """Linear layer with sin activation and SIREN weight initialisation."""

    def __init__(self, in_features: int, out_features: int, is_first: bool = False):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.is_first = is_first

        # Initialise weights to preserve signal statistics through the network.
        # First layer:   uniform(-1 / in_features,       1 / in_features)
        # Hidden layers: uniform(-sqrt(6 / in_features), sqrt(6 / in_features))
        ### STUDENT CODE HERE ###
        ...

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # The first layer scales its output by OMEGA_0 to set the base frequency.
        ### STUDENT CODE HERE ###
        ...


class SIREN(nn.Module):
    """Coordinate network with sinusoidal activations: 2 → hidden*depth → 3."""

    def __init__(self, hidden_size: int = 256, depth: int = 4):
        super().__init__()
        layers = [SIRENLayer(2, hidden_size, is_first=True)]
        for _ in range(depth - 1):
            layers.append(SIRENLayer(hidden_size, hidden_size))
        layers.append(nn.Linear(hidden_size, 3))  # final layer: no sin activation
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# 1. Train SIREN. Note the lower learning rate — SIREN is sensitive to large steps.
siren = SIREN(hidden_size=256, depth=4).to(device)
optimizer_siren = torch.optim.Adam(siren.parameters(), lr=1e-4)
siren_losses = []

for step in range(2000):
    pred = siren(coords)
    loss = nn.functional.mse_loss(pred, colors)
    optimizer_siren.zero_grad()
    loss.backward()
    optimizer_siren.step()
    siren_losses.append(loss.item())
    if step % 500 == 0:
        print(f"Step {step:4d} | Loss: {loss.item():.6f}")

# 2. Reconstruct and compare all three methods side by side.
siren.eval()
with torch.no_grad():
    recon_siren = siren(coords).clamp(0, 1).cpu().numpy().reshape(H, W, 3)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(img_np);           axes[0].set_title("Original");  axes[0].axis("off")
axes[1].imshow(naive_recon_saved); axes[1].set_title("Naive MLP"); axes[1].axis("off")
axes[2].imshow(pe_recon_saved);   axes[2].set_title("PE-MLP");    axes[2].axis("off")
axes[3].imshow(recon_siren);      axes[3].set_title("SIREN");     axes[3].axis("off")
plt.tight_layout()
plt.show()

siren_recon_saved = recon_siren.copy()  # kept for comparison in next cells

## Why SIREN Initialisation Matters

You just implemented the Sitzmann initialisation in Task 3.5. But what happens if you skip it and use PyTorch's default weight initialisation instead?

Run the cell below *without changes* and compare the result to Task 3.3.

**Question:** Why does the default initialisation cause SIREN to fail?

<details>
<summary><b>Click to see the answer</b></summary>
<br>

PyTorch initialises `Linear` layers with **Kaiming uniform** initialisation, designed for ReLU networks. For SIREN, this produces weights that are far too large. After the first `sin` activation, outputs saturate near ±1 everywhere — the network loses all spatial variation. Gradients then vanish and training stalls.

The Sitzmann scheme is derived to ensure that after each layer the distribution of activations remains approximately uniform on `[-1, 1]` throughout the network. This is the condition that keeps gradients alive and gives the network enough expressiveness to represent high-frequency content.

> Sitzmann, V. et al. (2020). *Implicit Neural Representations with Periodic Activation Functions*. NeurIPS. Sec. 3.2. [[arxiv](https://arxiv.org/abs/2006.09661)]

</details>

In [ ]:
torch.manual_seed(42)


# 1. Build a SIREN with PyTorch's default (Kaiming) initialisation — deliberately skipping the Sitzmann scheme.
class SIRENNoInit(nn.Module):
    """SIREN with default PyTorch weight initialisation."""

    def __init__(self, hidden_size: int = 256, depth: int = 4):
        super().__init__()
        layers = [nn.Linear(2, hidden_size)]
        for _ in range(depth - 1):
            layers.append(nn.Linear(hidden_size, hidden_size))
        layers.append(nn.Linear(hidden_size, 3))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = x
        for layer in self.net[:-1]:
            out = torch.sin(layer(out))
        return self.net[-1](out)


# 2. Train with default init.
siren_bad = SIRENNoInit(hidden_size=256, depth=4).to(device)
optimizer_bad = torch.optim.Adam(siren_bad.parameters(), lr=1e-4)
bad_losses = []

for step in range(2000):
    pred = siren_bad(coords)
    loss = nn.functional.mse_loss(pred, colors)
    optimizer_bad.zero_grad()
    loss.backward()
    optimizer_bad.step()
    bad_losses.append(loss.item())
    if step % 500 == 0:
        print(f"Step {step:4d} | Loss: {loss.item():.6f}")

# 3. Reconstruct and compare default init vs. Sitzmann init side by side.
siren_bad.eval()
with torch.no_grad():
    recon_bad = siren_bad(coords).clamp(0, 1).cpu().numpy().reshape(H, W, 3)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(img_np);            axes[0].set_title("Original");               axes[0].axis("off")
axes[1].imshow(recon_bad);         axes[1].set_title("SIREN (default init)");   axes[1].axis("off")
axes[2].imshow(siren_recon_saved); axes[2].set_title("SIREN (Sitzmann init)");  axes[2].axis("off")
plt.tight_layout()
plt.show()

print(f"Default init final loss:   {bad_losses[-1]:.6f}")
print(f"Sitzmann init final loss:  {siren_losses[-1]:.6f}")

## Task 3.4: Super-resolution

An INR is a **continuous function**, not a pixel grid. Once trained, you can query it at any resolution — including resolutions it never saw during training.

We trained all three models on a 128×128 grid. Let's query them at **256×256** and compare the quality to simply upsampling the original with PIL bicubic interpolation.

**Your job:** Build a 256×256 coordinate grid normalised to `[-1, 1]` and query each trained model.

**Question:** Which model super-resolves most faithfully, and why?

<details>
<summary><b>Click to see the answer</b></summary>
<br>

SIREN and PE-MLP generalise more smoothly to new coordinates because their inductive biases — sinusoidal activations and encoded frequencies — produce outputs that vary smoothly and richly between trained points. The naive ReLU MLP tends to produce blocky or blurry interpolation because piecewise-linear functions have no mechanism to extrapolate oscillatory detail.

This illustrates a core advantage of INRs over explicit representations: the continuous function learns a richer signal model than any upsampling filter can infer from a fixed pixel grid.

</details>

In [ ]:
H_sr, W_sr = 256, 256  # super-resolution target size

# Build a 256×256 coordinate grid normalised to [-1, 1].
### STUDENT CODE HERE ###
rows_sr = ...
cols_sr = ...
grid_y_sr, grid_x_sr = torch.meshgrid(rows_sr, cols_sr, indexing="ij")
coords_sr = torch.stack([..., ...], dim=-1).reshape(-1, 2).to(device)

# 1. Query all three trained models at the high-resolution coordinates.
naive_mlp.eval()
pe_mlp.eval()
siren.eval()
with torch.no_grad():
    recon_naive_sr = naive_mlp(coords_sr).clamp(0, 1).cpu().numpy().reshape(H_sr, W_sr, 3)
    recon_pe_sr    = pe_mlp(pe(coords_sr)).clamp(0, 1).cpu().numpy().reshape(H_sr, W_sr, 3)
    recon_siren_sr = siren(coords_sr).clamp(0, 1).cpu().numpy().reshape(H_sr, W_sr, 3)

# 2. Bicubic upsampling from PIL as a classical baseline.
img_bicubic = np.array(img_pil.resize((W_sr, H_sr), Image.BICUBIC), dtype=np.float32) / 255.0

# 3. Display the four results side by side.
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
titles = ["Bicubic (PIL)", "Naive MLP", "PE-MLP", "SIREN"]
images = [img_bicubic, recon_naive_sr, recon_pe_sr, recon_siren_sr]

for ax, title, img in zip(axes, titles, images):
    ax.imshow(img)
    ax.set_title(f"{title}\n(256×256)")
    ax.axis("off")

plt.suptitle("Super-resolution: trained at 128×128, queried at 256×256", fontsize=13)
plt.tight_layout()
plt.show()

## Task 3.5: Visual Comparison & PSNR

Let's make the comparison rigorous. **PSNR** (Peak Signal-to-Noise Ratio) is a standard metric for image quality. Higher is better (values above ~30 dB are considered good reconstruction).

The formula is:
```
PSNR = 10 · log₁₀(MAX² / MSE)
```
where `MAX = 1.0` for images normalised to `[0, 1]` and `MSE` is the mean squared error between the reconstruction and the original.

**Your job:** Fill in the PSNR function below.

**Question:** PSNR is popular but has known limitations. What is one thing it fails to capture?

<details>
<summary><b>Click to see the answer</b></summary>
<br>

PSNR is purely pixel-wise — it measures average squared error, but the human visual system does not perceive errors that way. A reconstruction can have a high PSNR yet look perceptually wrong (blurry, checkerboard artifacts) and vice versa.

A more perceptually-aligned metric is **SSIM** (Structural Similarity Index), which also considers luminance, contrast, and local structure. For the most perceptually accurate assessment, **LPIPS** (Learned Perceptual Image Patch Similarity) uses a pre-trained VGG network to compare feature representations.

</details>

In [ ]:
def psnr(original: np.ndarray, reconstruction: np.ndarray) -> float:
    """PSNR in dB for images with values in [0, 1]. Higher is better."""
    # Compute MSE, then return 10 * log10(1 / MSE).
    ### STUDENT CODE HERE ###
    mse = ...
    return ...


# 1. Print a PSNR table for all three methods.
results = {
    "Naive MLP": naive_recon_saved,
    "PE-MLP":    pe_recon_saved,
    "SIREN":     siren_recon_saved,
}

print(f"{'Method':<12} {'PSNR (dB)':>10}")
print("-" * 24)
for name, recon in results.items():
    print(f"{name:<12} {psnr(img_np, recon):>10.2f}")

# 2. Show the reconstructions side by side with their PSNR scores.
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(img_np); axes[0].set_title("Original"); axes[0].axis("off")
for ax, (title, img) in zip(axes[1:], results.items()):
    ax.imshow(img)
    ax.set_title(f"{title}\nPSNR: {psnr(img_np, img):.1f} dB")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Summary & Further Reading

Congratulations! You have gone from scratch to training three different neural representations of an image.

### What you learned?

| Concept | Key idea | Paper |
|---------|----------|-------|
| **Spectral bias** | Standard MLPs learn low frequencies first; fine details converge slowly | [Rahaman et al. 2019](https://arxiv.org/abs/1806.08734) |
| **INR** | A network as a continuous signal function: coordinates → values | [NeRF](https://arxiv.org/abs/2003.08934), [DeepSDF](https://arxiv.org/abs/1901.05103) |
| **Positional encoding** | Pre-process inputs with sinusoids at many frequencies to give the MLP "frequency handles" | [Tancik et al. 2020](https://arxiv.org/abs/2006.10739), [NeRF](https://arxiv.org/abs/2003.08934) |
| **SIREN** | Replace ReLU with `sin`; derivatives are also SIRENs; requires careful weight init | [Sitzmann et al. 2020](https://arxiv.org/abs/2006.09661) |
| **PSNR** | Standard pixel-level image quality metric (higher = better) | — |

### Where to go next?

> Feel free to experiment with different hyperparameters - lr, model sizes, etc. for better accuracy of different methods.

The field of INRs moves fast. Here are some directions that may be interesting for you! If not that's okay - we'll only use a handful of those during our experiments.

**More efficient training**
- [**Instant Neural Graphics Primitives** (Müller et al. 2022)](https://arxiv.org/abs/2201.05989) — replaces sinusoidal encoding with a multi-resolution hash table, making INR training 100–1000× faster. Powers many real-time NeRF systems.

**Alternative activation functions**
- [**WIRE** (Saragadam et al. 2023)](https://arxiv.org/abs/2301.05187) — uses complex Gabor wavelets as activations. Achieves higher PSNR than SIREN with fewer parameters.
- [**BACON** (Lindell et al. 2021)](https://arxiv.org/abs/2112.04645) — band-limited INRs that give explicit control over which frequencies the network represents.

**INRs in 3D**
- [**NeRF** (Mildenhall et al. 2020)](https://arxiv.org/abs/2003.08934) — combine an INR (density + colour as a function of 3D position and viewing direction) with differentiable volume rendering to reconstruct 3D scenes from 2D photos.
- [**3D Gaussian Splatting** (Kerbl et al. 2023)](https://arxiv.org/abs/2308.04079) — a non-network alternative that has largely replaced NeRF for real-time rendering.

**INRs beyond images**
- Audio signals, video, 3D occupancy fields, signed distance functions, weather and climate fields, and neural PDE solvers all use the same coordinate-network idea.

---

Now that you've finished this tutorial, you should be equiped with everything you need, to be able to recreate a single experiment from a research paper - the first task in our course. Remember to send me a proposal of which experiment you wanna do - on Slack!